In [9]:
# Load environment variables and create client

import json
from anthropic import Anthropic
from dotenv import load_dotenv
from building_with_the_claude_api import add_assistant_message, add_user_message, chat, Effort

load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5"

In [10]:
# Generate evaluation dataset

def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
    each representing task that requires Python, JSON, or a Regex to complete.
    The format should be: python, json, or regex.
    Add a solution criteria to help grade solutions to this task.

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """

    tasks_schema = {
        "type": "array",
        "items": {
            "type": "object",
            "properties": {
                "task": {"type": "string"},
                "format": {"type": "string"},
                "solution_criteria": {"type": "string"},
            },
            "required": ["task", "format", "solution_criteria"],
            "additionalProperties": False,
        },
    }

    # Structured data: generate N macOS CLI commands as JSON
    messages = []

    add_user_message(messages=messages, text=prompt)
    text = chat(messages=messages, client=client, json_schema=tasks_schema, model=model, effort=Effort.NONE)

    result = json.loads(text)
    with open("../../tmp/eval_dataset.json", "w") as f:
        json.dump(result, f, indent=2)


# call method
generate_dataset()

In [11]:
def run_prompt(test_case):
    """ Merges the prompt and test case and return the result"""

    prompt = f"""
    Please solve the following task:
    {test_case['task']}

    Solution criteria:
    {test_case['solution_criteria']}

    * Response only with Python, JSON, or a plain Regexp
    * Do not add any comments, commentary or explanation
    """

    code_schema = {
        "type": "object",
        "properties": {
            "code": {"type": "string"},
        },
        "required": ["code"],
        "additionalProperties": False,
    }

    messages = []
    add_user_message(messages=messages, text=prompt)
    output = chat(messages=messages, client=client, model=model, effort=Effort.NONE, json_schema=code_schema)
    return json.loads(output)["code"]

In [12]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to Evaluate:
    <solution>
    {output}
    </solution>

    Solution criteria:
    <Criteria>
    {test_case["solution_criteria"]}
    </Criteria>

    Output Format
    Provide your evaluation as a structured JSON object with the following fields, in this specific order:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your overall assessment
    - "score": A number between 1-10

    Keep your response concise and direct.
    """

    eval_schema = {
        "type": "object",
        "properties": {
            "strengths": {"type": "array", "items": {"type": "string"}},
            "weaknesses": {"type": "array", "items": {"type": "string"}},
            "reasoning": {"type": "string"},
            "score": {"type": "number"},
        },
        "required": ["strengths", "weaknesses", "reasoning", "score"],
        "additionalProperties": False,
    }

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages=messages, client=client, json_schema=eval_schema, model=model, effort=Effort.NONE)
    return json.loads(eval_text)

In [13]:
# Functions to validate the output structure
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [14]:
def run_test_case(test_case):
    """ Calls run_prompt then grade the result"""
    output = run_prompt(test_case)

    # grading
    model_grade = grade_by_model(test_case, output)
    syntax_score = grade_syntax(output, test_case)
    score = (model_grade["score"] + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "model_grade": model_grade,
        "syntax_score": syntax_score,
    }

In [15]:
from statistics import mean

def run_eval(dataset):
    """ Loads the dataset and calls run_test_case for each case"""
    results = []

    for test_case in dataset:
        results.append(run_test_case(test_case))

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [17]:
with open("../../tmp/eval_dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

Average score: 6.833333333333333
[
  {
    "output": "import re\n\ndef parse_s3_arn(arn):\n    pattern = r'^arn:aws:s3:::([a-z0-9][a-z0-9\\-_]*[a-z0-9])$'\n    match = re.match(pattern, arn)\n    if match:\n        return match.group(1)\n    return None\n\ntest_cases = [\n    ('arn:aws:s3:::my-bucket', 'my-bucket'),\n    ('arn:aws:s3:::invalid.bucket', None),\n    ('arn:aws:s3:::my_bucket', 'my_bucket'),\n    ('arn:aws:s3:::my-bucket-name', 'my-bucket-name')\n]\n\nfor arn, expected in test_cases:\n    result = parse_s3_arn(arn)\n    assert result == expected, f'Failed for {arn}: got {result}, expected {expected}'\n\nprint('All tests passed')",
    "test_case": {
      "task": "Parse an AWS S3 bucket name and region from a CloudFormation ARN string like 'arn:aws:s3:::my-bucket-name'",
      "format": "regex",
      "solution_criteria": "The regex should correctly extract bucket names containing hyphens and underscores, handle optional region information, and not match invalid ARN format